In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from backend.app.services.data_services import (
    get_available_indicators,
    get_available_strategies,
    get_strategies_metadata,
    get_available_tickers,
    _read_csv,
    fetch_data_to_df,
    get_indicators_metadata,
    _load_and_resample_data
)
import importlib
import backend.app.core.Strategies as st

import backend.app.services.cache_service as cs
import backend.app.services.data_services as ds
import backend.app.services.plotting_services as ps
import backend.app.services.backtest_services as bs

# importlib.reload(cs)
# importlib.reload(ds)
# importlib.reload(ps)
# importlib.reload(bs)

import asyncio
import pandas as pd
import numpy as np
import vectorbt as vbt
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import webbrowser
import os

from backend.app.core.Strategies import STRATEGY_REGISTRY
from backend.app.core.Indicators import INDICATOR_REGISTRY

In [3]:
ticker_name = "BTCUSD"
df = await ds._read_csv(ticker_name=ticker_name)
print(df.head(2))
print(df.tail(2))
await cs.set_data(df=df, ticker= ticker_name)

                        open     high      low    close  volume
date_time                                                      
2025-01-01 00:00:00  93968.0  94040.5  93929.5  94040.5    7873
2025-01-01 00:01:00  94040.5  94125.5  94039.0  94125.5   12929
                        open     high      low    close  volume
date_time                                                      
2025-12-20 23:59:00  88174.5  88174.5  88173.5  88174.5     427
2025-12-21 00:00:00  88174.5  88174.5  88173.5  88174.5     362


{'message': 'Data loaded successfully with key: data:BTCUSD'}

In [4]:
resolution = "15m"
start_date = "01/05/2025 09:15:00"
end_date = "30/10/2025 09:15:00"

df = await ds._load_and_resample_data(ticker_name, resolution, start_date, end_date)
print(df.shape)
price = df['close']
df.head(2)

d:\OneDrive - iitgn.ac.in\Desktop\HedgeOne-Quant\backend\app\services\data_services.py:54: FutureWarning:

'T' is deprecated and will be removed in a future version, please use 'min' instead.



(17473, 5)


,open,high,low,close,volume
2025-05-01 09:15:00,94792.5,94793.0,94664.0,94737.5,33436
2025-05-01 09:30:00,94738.0,94846.0,94702.5,94779.0,29671


In [5]:
ema_3 = INDICATOR_REGISTRY["Exponential Moving Average (EMA)"](df, **{"window":3})["EMA"]
ema_6 = INDICATOR_REGISTRY["Exponential Moving Average (EMA)"](df, **{"window":6})["EMA"]
rsi = INDICATOR_REGISTRY["Relative Strength Index (RSI)"](df, **{"window":15})["RSI"]

In [6]:
st  = STRATEGY_REGISTRY["EMA Crossover"](df,**{"short_window":3, "long_window":6})

entries = st["entries"] & (np.array(rsi)>75)
exits = st["exits"]

In [7]:
p = vbt.Portfolio.from_signals(
    close=price,
    entries=entries,
    exits=exits,
    init_cash=100000,
    fees=0.0,
    freq=resolution,
    # sl_stop=sl_dist,
    # tp_stop=tp_dist,
    # sl_trail=tsl_dist,
    # accumulate=False
)

In [8]:
print(round(p.total_return()*100, 2), round(p.stats()["Win Rate [%]"], 2), round(p.sharpe_ratio(),2), p.stats()["Total Trades"])

0.63 50.0 0.79 6


In [9]:
p.plot()

FigureWidget({
    'data': [{'legendgroup': '0',
              'line': {'color': '#1f77b4'},
              'name': 'Close',
              'showlegend': True,
              'type': 'scatter',
              'uid': '8f7a0c51-4d79-48ea-9556-64493e9d26d0',
              'x': array(['2025-05-01T09:15:00.000000000', '2025-05-01T09:30:00.000000000',
                          '2025-05-01T09:45:00.000000000', ..., '2025-10-30T08:45:00.000000000',
                          '2025-10-30T09:00:00.000000000', '2025-10-30T09:15:00.000000000'],
                         dtype='datetime64[ns]'),
              'xaxis': 'x',
              'y': {'bdata': ('AAAAABgh90AAAAAAsCP3QAAAAADYHf' ... 'AAaAn7QAAAAADQBftAAAAAADgF+0A='),
                    'dtype': 'f8'},
              'yaxis': 'y'},
             {'customdata': {'bdata': ('AAAAAAAAAACBkNm+50XtPwAAAAAAAA' ... 'AAAAAAJEBavRhDgF3sPwAAAAAAAAAA'),
                             'dtype': 'f8',
                             'shape': '6, 3'},
              'hove